In [6]:
import requests
import copy
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from sqlalchemy import text
import os
import sys

projectRoot = os.path.abspath(os.path.join(os.getcwd(), '..'))
if projectRoot not in sys.path:
    sys.path.insert(0, projectRoot)
from helpers import initDB

<div class='alert alert-block alert-info'>
<b>Team Games:</b> using a team's link, all their games for the year will be requested and parsed
</div>

In [7]:
#### direct graphql request ####
from graphqlQueries import teamFixtureQuery

def directTeamRequest(teamURL, session=None):
    uniqueSession = session is None
    if uniqueSession:
        session = requests.Session()
    graphqlURL = 'https://api.playhq.com/graphql'
    teamID = teamURL.rstrip('/').split('/')[-1]
    headers = {
        "content-type": "application/json",
        "origin": "https://www.playhq.com",
        "referer": teamURL,
        "tenant": "afl",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    }
    payload = {
        "operationName": "teamFixture",
        "variables": {"teamID": teamID},
        "query": teamFixtureQuery
    }
    try:
        response = session.post(graphqlURL, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()['data']
    finally:
        if uniqueSession:
            session.close()
    
#directTeamRequest('https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-c3/bb882c71')

In [8]:
def parseTeamData(data):
    grade = data['discoverTeam']['grade']['name'].lower().replace(' ', '-')
    rounds = data['discoverTeamFixture']
    gameIDs = []
    for round in rounds:
        if round['fixture']['byes'] == []:
            roundGameID = round['fixture']['games'][0]['id']
            gameIDs.append(roundGameID)
    return {'grade': grade, 'games': gameIDs}

#parseTeamData(data)

<div class='alert alert-block alert-info'>
<b>Game Players:</b> using a list of game links, all particpating players will be requested and parsed
</div>

In [9]:
from graphqlQueries import gamePlayersQuery

def directGameRequest(url, session=None):
    uniqueSession = session is None
    if uniqueSession:
        session = requests.Session()
    graphqlURL = 'https://api.playhq.com/graphql'
    gameID = url.rstrip('/').split('/')[-1]
    headers = {
        "content-type": "application/json",
        "origin": "https://www.playhq.com",
        "referer": url,
        "tenant": "afl",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    }
    payload = {
        "operationName": "gameView",
        "variables": {
            "gameId": gameID,
            "gameStatisticsFilter": {
                "classification": "TOTAL"
            }
        },
        "query": gamePlayersQuery
    }
    try:
        response = session.post(graphqlURL, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()['data']
    finally:
        if uniqueSession:
            session.close()

#directGameRequest('https://www.playhq.com/afl/org/perth-football-league/perth-football-league-2025/budget-car-and-truck-rental-c3-men/game-centre/fab3fdd5')

In [10]:
def parseGameData(data):
    data = data['discoverGame']
    side = 'home'
    if data['home']['organisation']['name'] != 'University (Perth Football League)':
        side = 'away'
    players = data['statistics'][side]['players']
    scrapedPlayers = []
    for player in players:
        player = player['player']
        if 'name' in player:
            pass #private player
        else:
            player = player['profile']
            scrapedPlayers.append([player['id'], player['firstName'], player['lastName']])
    return scrapedPlayers

#parseGameData(gameData)

<div class='alert alert-block alert-info'>
<b>Player Career:</b> given a player's unique link, their university playing history will be requested and parsed
</div>

In [11]:
from graphqlQueries import playerHistoryQuery

def directPlayerRequest(url, session=None):
    uniqueSession = session is None
    if uniqueSession:
        session = requests.Session()
    graphqlURL = 'https://api.playhq.com/graphql'
    profileID = url.rstrip('/').split('/')[-2]
    headers = {
        "content-type": "application/json",
        "origin": "https://www.playhq.com",
        "referer": "https://www.playhq.com/",
        "tenant": "afl",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    }
    payload = {
        "operationName": "publicProfileStatistics",
        "variables": {
            "profileID": profileID
        },
        "query": playerHistoryQuery
    }
    try:
        response = session.post(graphqlURL, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()['data']
    finally:
        if uniqueSession:
            session.close()

#directPlayerRequest('https://www.playhq.com/public/profile/490d2f42-daea-43ae-8bf1-b10b0e2cab2b/statistics?tenant=afl')

In [12]:
def parseProfileData(data):
    clubs = data['publicProfileStatistics']['careerStatistics']['clubStatistics']
    for club in clubs:
        if club['club']['name'] == 'University (Perth Football League)':
            playerStats = club['statistics']
            for stat in playerStats:
                if stat['details']['value'] == 'APPEARANCE':  
                    return int(stat['count'])
    profile = data['publicProfile']
    print(f"Could not find any data for {profile['firstName']} {profile['lastName']} with ID: {profile['id']}")
    return 0

#parseProfileData(profileData)

<div class='alert alert-block alert-info'>
<b>Club Players:</b> given a list of team links, all participating players will have their url listed
</div>

In [13]:
def clubUniquePlayers(teamLinks):
    uniquePlayers = {}
    with requests.Session() as session:
        for teamLink in teamLinks:
            print(f'Requesting {teamLink}...')
            #locate all games a team partakes in
            rawTeamInfo = directTeamRequest(teamLink, session=session)
            teamInfo = parseTeamData(rawTeamInfo)
            teamGrade = teamInfo['grade']
            teamGames = teamInfo['games']
            for idx, game in enumerate(teamGames):
                print(f'{idx+1}/{len(teamGames)}')
                #locate all uni players in each game
                gameLink = f'https://www.playhq.com/afl/org/perth-football-league/perth-football-league-2025/{teamGrade}/game-centre/{game}'
                rawGameInfo = directGameRequest(gameLink, session=session)
                gameInfo = parseGameData(rawGameInfo)
                for playerID, firstName, lastName in gameInfo:
                    #storing each player within the global dict
                    uniquePlayers[playerID] = [playerID, firstName, lastName]
    return list(uniquePlayers.values())
            
    
#clubPlayers = clubUniquePlayers(teamLinks) 
#requests 3m27.3s
#session+!deepCopy 1m44.7s

def parallelClubUniquePlayers(teamLinks, maxWorkers=8):
    uniquePlayers = {}
    gameLinks = []
    #locating all games 
    with requests.Session() as session:
        for teamIdx, teamLink in enumerate(teamLinks, start=1):
            print(f'Team {teamIdx}/{len(teamLinks)}')
            rawTeamInfo = directTeamRequest(teamLink, session=session)
            teamInfo = parseTeamData(rawTeamInfo)
            teamGrade = teamInfo['grade']
            teamGames = teamInfo['games']
            for game in teamGames:
                gameLink = f'https://www.playhq.com/afl/org/perth-football-league/perth-football-league-2025/{teamGrade}/game-centre/{game}'
                gameLinks.append(gameLink)
    print(f'Queued {len(gameLinks)} games...')
    #finding all university players who participated in matches
    with ThreadPoolExecutor(max_workers=maxWorkers) as executor:
        futures = {
            executor.submit(directGameRequest, gameLink): gameLink for gameLink in gameLinks #mustn't share session across threads
        }
        for idx, future in enumerate(as_completed(futures), start=1):
            gameLink = futures[future]
            print(f'Game {idx}/{len(gameLinks)}')
            rawGameInfo = future.result()
            gameInfo = parseGameData(rawGameInfo)
            for playerID, firstName, lastName in gameInfo:
                uniquePlayers[playerID] = [playerID, firstName, lastName]
    return list(uniquePlayers.values())

#38.8s
#clubPlayers = parallelClubUniquePlayers(teamLinks2025, maxWorkers=8) 

<div class='alert alert-block alert-info'>
<b>Player Aggregates:</b> using a list of player URLs, return a list of player details including their total university game count
</div>

In [14]:
def playerCounts(clubPlayers):
    players = copy.deepcopy(clubPlayers) #for testing, remove in prod
    uniquePlayers = {}
    for idx, player in enumerate(players):
        print(f'{idx+1}/{len(players)}')
        playerID, firstName, lastName = player
        rawData = directPlayerRequest(f'https://www.playhq.com/public/profile/{playerID}/statistics?tenant=afl')
        count = parseProfileData(rawData)
        uniquePlayers[playerID] = [playerID, firstName, lastName, count]
    return list(uniquePlayers.values())

#completePlayers = playerCounts(clubPlayers) #4m57s

def parallelPlayerCounts(clubPlayers, maxWorkers=8):
    uniquePlayers = {}
    with ThreadPoolExecutor(max_workers=maxWorkers) as executor:
        futures = {
            executor.submit(
                directPlayerRequest, f'https://www.playhq.com/public/profile/{playerID}/statistics?tenant=afl'
            ) : (playerID, firstName, lastName) 
            for playerID, firstName, lastName in clubPlayers
        }
        for idx, future in enumerate(as_completed(futures), start=1):
            playerID, firstName, lastName = futures[future]
            print(f'{idx}/{len(clubPlayers)}')
            rawData = future.result()
            count = parseProfileData(rawData)
            uniquePlayers[playerID] = [playerID, firstName, lastName, count]
    return list(uniquePlayers.values())
#1m43.2s
#parallelPlayers = parallelPlayerCounts(clubPlayers, maxWorkers=8)

<div class='alert alert-block alert-success'>
<b>Scrape Club:</b> request, parse and save all current university players in a database 
</div>

In [15]:
teamLinks2024 = [
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-a/341833cd',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-ar/ee8e0ccb',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-bjc/67e45bef',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-c4/9e89ed49',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-c4r/96ecd53e',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-psc/e064c5d3',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-arw/327fa075',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2024/teams/university-wa/ec5f13f7'
]

teamLinks2025 = [
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-a/a7101275',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-ar/f63bf5dd',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-bjc/ae04ae10',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-c3/bb882c71',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-c3r/89ad8e8e',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-psc/807f5521',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-arw/ec91f6cc',
    'https://www.playhq.com/afl/org/university-perth-football-league/97522c61/perth-football-league-2025/teams/university-wa/d5496ec8',
]

In [ ]:
def scrapeClub(clubLinks):
    teamLinks = [link for seasonLinks in clubLinks for link in seasonLinks]
    engine = initDB()
    with engine.connect() as conn:
        existingDF = pd.read_sql(text('SELECT * FROM players'), conn)
    clubPlayers = parallelClubUniquePlayers(teamLinks, maxWorkers=8)
    players = parallelPlayerCounts(clubPlayers, maxWorkers=8)
    newDF = pd.DataFrame(players, columns=['ID', 'FirstName', 'LastName', 'GameCount'])
    df = pd.concat([existingDF, newDF]).drop_duplicates(subset='ID').reset_index(drop=True)
    df = df.sort_values(by='GameCount', ascending=False)
    df.to_sql('players', con=engine, if_exists='replace', index=False)
    print(f'Players Added: {len(df.index)-len(existingDF.index)}')
    print(f'Players Updated: {len(newDF.index)}')
    return df
    
#3m9.5s
scrapeClub([teamLinks2024, teamLinks2025])
#TO DO: update for season in progress as game scrape may break on games not yet complete

Team 1/16
Team 2/16
Team 3/16
Team 4/16
Team 5/16
Team 6/16
Team 7/16
Team 8/16
Team 9/16
Team 10/16
Team 11/16
Team 12/16
Team 13/16
Team 14/16
Team 15/16
Team 16/16
Queued 302 games...
Game 1/302
Game 2/302
Game 3/302
Game 4/302
Game 5/302
Game 6/302
Game 7/302
Game 8/302
Game 9/302
Game 10/302
Game 11/302
Game 12/302
Game 13/302
Game 14/302
Game 15/302
Game 16/302
Game 17/302
Game 18/302
Game 19/302
Game 20/302
Game 21/302
Game 22/302
Game 23/302
Game 24/302
Game 25/302
Game 26/302
Game 27/302
Game 28/302
Game 29/302
Game 30/302
Game 31/302
Game 32/302
Game 33/302
Game 34/302
Game 35/302
Game 36/302
Game 37/302
Game 38/302
Game 39/302
Game 40/302
Game 41/302
Game 42/302
Game 43/302
Game 44/302
Game 45/302
Game 46/302
Game 47/302
Game 48/302
Game 49/302
Game 50/302
Game 51/302
Game 52/302
Game 53/302
Game 54/302
Game 55/302
Game 56/302
Game 57/302
Game 58/302
Game 59/302
Game 60/302
Game 61/302
Game 62/302
Game 63/302
Game 64/302
Game 65/302
Game 66/302
Game 67/302
Game 68/302
Game 6

,ID,FirstName,LastName,GameCount
0,da5aadfb-0257-4b38-bb47-ee09eca01efb,Greg,Erskine,283
1,22354897-3648-45c9-ace2-57d08bc0d724,Harry,Donovan,252
2,94f7381d-c4f8-4ee4-a3c5-5ed8d66ffea9,Patrick,Donovan,249
414,01ebf085-b466-413e-9392-7d8c18edaa64,Felix,Hudson,246
3,1632d2eb-dbef-4c1f-9a84-713d8ea895db,Tim,O’Hara,224
...,...,...,...,...
325,22e5ae72-cdd3-4e4b-9876-efe44c19e5d5,Sam,Pierce,1
431,24162e7b-4c3d-41ed-9e92-af536dfb903c,Regan,Mott,1
424,aebde607-f25f-4058-bff7-302d1a93a747,Joshua,Francis,1
455,5b2fe5c2-2c0c-4c87-95cc-dcd9b890005c,Jemma,Brook,1
